# Mi primer recorrido de clasificación: Titanic

Continuamos el EDA: ahora queremos **predecir**, no demostrar qué causó la supervivencia.
Trabajad por parejas. Cada persona ejecuta, explica y anota los checkpoints.
Usad el CSV original `titanic.csv` junto a este cuaderno. No se descarga nada.

El EDA anterior se hizo sobre el conjunto completo: esta evaluación es una **primera demostración**, no una estimación completamente independiente. Para un proyecto real reservaríamos prueba antes de explorar y elegir características.

## 1. Pregunta, datos y variables

¿Podemos predecir `Survived` usando algunas características conocidas? Decid por qué
`Survived` y su posible copia `Survived_label` **no** pueden entrar en X. El CSV no se modifica.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

ruta = Path("titanic.csv")
if not ruta.is_file():
    raise FileNotFoundError("Coloca titanic.csv junto al cuaderno y vuelve a ejecutar.")
df = pd.read_csv(ruta)
print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__)
print("Tamaño:", df.shape)
display(df.head(3))
assert df.shape == (891, 12)
assert set(df["Survived"].unique()) == {0, 1}


In [ ]:
# Pclass se trata como categoría aunque esté representada con números.
numericas = ["Age", "Fare", "SibSp", "Parch"]
categoricas = ["Pclass", "Sex", "Embarked"]
X = df[numericas + categoricas].copy()
y = df["Survived"].copy()
assert {"Survived", "Survived_label"}.isdisjoint(X.columns)
print("Características:", X.columns.tolist())
print("Objetivo:", y.name, "| filas:", len(y))
print("Valores ausentes por columna:")
print(X.isna().sum())


**Checkpoint 1:** Explica con tus palabras qué son X e y y por qué no incluimos la respuesta en X.
Anota una característica excluida y una razón. La selección es una simplificación, no una regla universal.

## 2. Separamos antes de aprender transformaciones

Reservamos el 20 % para prueba. Estratificar mantiene aproximadamente las proporciones
de las clases. La semilla permite repetir esta partición bajo el mismo entorno.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
assert set(X_train.index).isdisjoint(X_test.index)
assert len(X_train) + len(X_test) == len(df)
print("Entrenamiento:", X_train.shape, "| prueba:", X_test.shape)
print("Supervivencia en train:", round(y_train.mean(), 3))


**Checkpoint 2:** ¿Qué conjunto se usa para aprender? ¿Para qué reservamos el otro?
No calcules medianas, escalas o categorías sobre todo el archivo antes de separar.

## 3. Preparación y modelo juntos

Nulos numéricos: mediana; categóricos: valor más frecuente. Codificamos categorías
y escalamos los números. `Pipeline` ajusta preparación y clasificador solo con train.
La configuración está fijada antes de consultar prueba.

### Dos clasificadores, dos funciones

`DummyClassifier` sirve de **referencia**: predice la clase más frecuente de
entrenamiento sin usar las características. `LogisticRegression` aprende
coeficientes de las características preparadas y estima una probabilidad para
clasificar. Su nombre contiene «regresión», pero aquí resuelve una clasificación
binaria. La exactitud no explica por sí sola qué tipos de errores comete.

Para ampliar, consulta `05b-primeros-clasificadores.md` si el profesor facilita
la guía; en este primer recorrido basta con explicar qué aprende cada uno.


In [ ]:
preparacion = ColumnTransformer([
    ("numericas", Pipeline([
        ("imputar", SimpleImputer(strategy="median")),
        ("escalar", StandardScaler()),
    ]), numericas),
    ("categoricas", Pipeline([
        ("imputar", SimpleImputer(strategy="most_frequent")),
        ("codificar", OneHotEncoder(handle_unknown="ignore")),
    ]), categoricas),
])
modelo = Pipeline([
    ("preparar", preparacion),
    ("clasificar", LogisticRegression(max_iter=1000, random_state=42)),
])
base = DummyClassifier(strategy="most_frequent")
base.fit(X_train, y_train)
modelo.fit(X_train, y_train)
print("Preparación y modelo ajustados únicamente con entrenamiento.")


### Observar lo que preparó el pipeline

Esta comprobación incorpora la preparación para ML del apartado 4 de la sesión
Titanic de UD2: **no hay que abrir aquel cuaderno después de este**.
El pipeline ya aprendió las medianas, escalas y categorías al entrenar.
Ahora solo usamos `transform` para inspeccionar cómo deja train y test, sin
volver a aprender nada. Las columnas nuevas proceden de la codificación.


In [ ]:
preparador_ajustado = modelo.named_steps["preparar"]
train_preparado = preparador_ajustado.transform(X_train)
test_preparado = preparador_ajustado.transform(X_test)
print("Train preparado:", train_preparado.shape)
print("Test preparado:", test_preparado.shape)
print("Columnas:", preparador_ajustado.get_feature_names_out().tolist())
assert train_preparado.shape[0] == len(X_train)
assert test_preparado.shape[0] == len(X_test)
assert train_preparado.shape[1] == test_preparado.shape[1]
for tabla in (train_preparado, test_preparado):
    valores = tabla.toarray() if hasattr(tabla, "toarray") else np.asarray(tabla)
    assert np.isfinite(valores).all()


**Checkpoint 3:** ¿De qué pasajeros aprende la mediana? ¿Qué hace el pipeline
cuando llega un pasajero de prueba con una categoría no vista?

## 4. Predicción base, modelo y errores

La base predice siempre la clase más frecuente de train. Ambos métodos se evalúan
en los mismos pasajeros reservados. La matriz tiene **filas reales y columnas predichas**;
0 = no sobrevivió, 1 = sobrevivió.

In [ ]:
pred_base = base.predict(X_test)
pred = modelo.predict(X_test)
print("Exactitud de la base:", round(accuracy_score(y_test, pred_base), 3))
print("Exactitud del modelo:", round(accuracy_score(y_test, pred), 3))
matriz = confusion_matrix(y_test, pred, labels=[0, 1])
display(pd.DataFrame(matriz, index=["Real 0", "Real 1"],
                     columns=["Predicción 0", "Predicción 1"]))
assert matriz.sum() == len(y_test)


**Checkpoint 4:** Localiza un falso positivo y un falso negativo en la matriz.
¿Qué conclusión permite la comparación con la base? ¿Por qué la exactitud sola
no basta para garantizar que el modelo sea útil o justo?

No ajustéis el modelo mirando repetidamente test. Para comparar modificaciones,
reservaremos validación dentro de train en una sesión posterior.

In [ ]:
ejemplos = X_test.head(5).copy()
ejemplos["resultado_real"] = y_test.loc[ejemplos.index]
ejemplos["prediccion"] = modelo.predict(X_test.head(5))
display(ejemplos)


## 5. Pequeño trabajo propio y cierre

1. Sin reentrenar, localiza en la tabla o matriz un acierto y un tipo de error;
   explica qué significan.
2. Propón una característica alternativa y explica por qué podría o no ser útil.
   **No vuelvas a consultar test para decidir si mejora.**
3. En tu cuaderno de seguimiento anota: hecho / pendiente; qué se aprende de train,
   para qué se reserva test, una limitación y la aportación de cada persona.

No se crea una nueva entrega formal ni una nota por este primer recorrido.
Sigue las indicaciones del profesor para compartir tu trabajo.
